# 01 Data understanding

**Purpose.** Describe the data before any modelling: what one record is, how candidates, labels and histories are distributed, what the anonymised text allows, and which properties of the files the pipeline has to handle (file order, missing metadata, time gaps). It also gives a feature dictionary with descriptive statistics.

**Scope.** Raw-data summaries use all 173,831 records and the full catalogue. Engineered-feature statistics use a uniform 5,000-record sample (seed 4222) with all candidates and all history of each record; notebook 02 keeps these records out of validation and test. The raw files are streamed into compact summaries; no full feature table is built.

**Outputs** (working folder, `outputs/data_understanding/`): tables (`tables/*.csv`) and figures (`figures/*.png`).

**Runtime:** high-RAM CPU runtime, about 10 minutes.

## 1. Business problem and ML task

**Business context.** JDsearch provides a current query, the candidate products JD showed for it, and the shopper's supplied interaction history. Historical preferences may help the current search, but how relevant a past behaviour is can vary with the query and the candidate.

- **Business objective:** help shoppers find suitable products sooner by using their history where it is useful evidence for the current query, without letting past favourites crowd out what the query asks for. Offline ranking quality is the measurable proxy.
- **ML task:** personalised re-ranking of the supplied candidate products of each search.
- **Inputs:** the current query, the supplied history (products, action types, historical queries and relative time gaps), and candidate metadata (title terms, brand, four category levels, shop).
- **Input sample and grouping:** a (search, candidate) pair scored in the context of its complete search record. All candidates of one search form a ranking group and stay together when sampling or splitting.
- **Target:** the observed interaction grade of each candidate: 0 = no interaction, 1 = click, 2 = add to cart, 3 = purchase. These labels supervise the ranking; they do not measure monetary value.
- **Evaluation:** full, un-downsampled candidate lists; NDCG@10 with linear gains as the primary metric, Recall@10 and MRR as support (notebook 02).

**Why ranking metrics, not accuracy.** About 98% of candidate labels are zero (section 4.2), so predicting no interaction everywhere gives high accuracy without any useful ranking.

## 2. Setup

In [ ]:
# Setup: on Colab, clone the repository (shared code in jdsearch.py) and use Google Drive as the working folder.
import subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/repo").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/liamkunert/BT4222-group17-jdsearch-submission.git", "/content/repo"], check=True)
    sys.path.insert(0, "/content/repo")
else:
    sys.path.insert(0, str(Path.cwd().resolve().parent))
import jdsearch as jd

paths = jd.setup()
jd.require(paths, "data/user_behavior_data.txt", "data/product_meta_data.txt")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

OUT = paths.outputs / "data_understanding"
FIG_DIR, TABLE_DIR = OUT / "figures", OUT / "tables"
for folder in (FIG_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
SEED, EDA_SAMPLE_SIZE = 4222, 5000
jd.validate()
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
MUTED = "#898781"
plt.rcParams.update({"figure.figsize": (9, 4), "figure.dpi": 110, "savefig.dpi": 180,
                    "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False,
                    "axes.grid": True, "grid.color": "#e6e5e1", "axes.axisbelow": True,
                    "axes.titleweight": "bold", "axes.titlelocation": "left"})
pd.options.display.float_format = "{:,.4f}".format
pd.options.display.max_colwidth = 100


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")


def save_table(frame, name, scope):
    result = frame.copy()
    result["scope"] = scope
    result.to_csv(TABLE_DIR / f"{name}.csv")
    return result


FULL_SCOPE = "full behaviour file / full catalog as applicable"
SAMPLE_SCOPE = f"{EDA_SAMPLE_SIZE:,} complete records; seed {SEED}"

### Helper functions

Two streaming summaries: `build_full_profile` reads both raw files once and keeps only compact arrays and counters (validating list alignment, labels, the initial time placeholder and non-negative gaps on the way); `build_sample_features` computes example features on the complete records of a sample. `numeric_stats`, `describe_frequency` and `compact_lorenz` summarise distributions.

In [ ]:
from array import array
from collections import Counter

from jdsearch import LEVELS, SOURCES, match_counts, overlap_features, token_set, tokenize


def numeric_stats(frame):
    return pd.DataFrame({"min": frame.min(), "max": frame.max(), "mean": frame.mean(),
                         "median": frame.median(), "std": frame.std(), "missing": frame.isna().sum(),
                         "observed": frame.notna().sum()})


def describe_frequency(counts):
    """Exact descriptive statistics from an integer-value frequency table."""
    keys = np.asarray(sorted(counts), dtype=np.int64)
    freq = np.asarray([counts[int(key)] for key in keys], dtype=np.int64)
    n = int(freq.sum())
    if not n:
        return pd.Series(dtype=float)
    cumulative = np.cumsum(freq)
    mean = np.dot(keys.astype(float), freq) / n
    def quantile(p):
        location = (n - 1) * p
        low, high = int(np.floor(location)), int(np.ceil(location))
        a, b = keys[np.searchsorted(cumulative, [low + 1, high + 1])]
        return float(a + (b - a) * (location - low))
    variance = np.dot((keys - mean) ** 2, freq) / (n - 1) if n > 1 else np.nan
    return pd.Series({"count": n, "min": keys[0], "max": keys[-1], "mean": mean,
                      "median": quantile(.5), "std": np.sqrt(variance), "p25": quantile(.25), "p75": quantile(.75), "p95": quantile(.95)})


def members(sorted_ids, ids):
    indices = np.minimum(np.searchsorted(sorted_ids, ids), len(sorted_ids) - 1)
    return sorted_ids[indices] == ids


def compact_lorenz(counts):
    ordered = np.sort(counts)[::-1]
    cumulative = np.cumsum(ordered)
    stops = np.unique(np.r_[0, np.linspace(1, len(ordered), 1001).astype(int)])
    shares = np.r_[0., cumulative[stops[1:] - 1] / cumulative[-1]]
    return stops / len(ordered), shares


def build_full_profile(behaviour, catalogue, processed):
    """Full-data summaries of both raw files, streamed without keeping their text or a candidate feature table."""
    candidate_buffer, history_buffer, gap_buffer = array("q"), array("q"), array("q")
    label_buffer = array("b")
    session_rows, raw_queries = [], Counter()
    actions, queryless_actions, text_counts, text_chars = Counter(), Counter(), {}, {}
    excluded_text = Counter()
    query_vocabulary = set()
    label_session_counts = np.zeros(4, dtype=np.int64)
    threshold_session_counts = np.zeros(4, dtype=np.int64)
    positive_prefix_records, positive_records = 0, 0
    def text_observation(field, value):
        if value in ("", "-1"):
            excluded_text[field] += 1
            return
        text_counts.setdefault(field, Counter())[value.count("\x18") + 1] += 1
        text_chars.setdefault(field, Counter())[len(value)] += 1
    with Path(behaviour).open(encoding="utf-8") as handle:
        header = next(handle).rstrip("\r\n").split("\t")
        assert header == ["query", "candidate_wid_list", "candidate_label_list", "history_qry_list", "history_wid_list", "history_type_list", "history_time_list"]
        for session_id, line in enumerate(handle):
            fields = line.rstrip("\r\n").split("\t")
            assert len(fields) == 7
            candidates = np.fromstring(fields[1], sep="_", dtype=np.int64)
            raw_labels = np.fromstring(fields[2], sep="_", dtype=float)
            labels = raw_labels.astype(np.int8)
            histories = np.fromstring(fields[4], sep="_", dtype=np.int64)
            history_queries, history_actions = fields[3].split("_"), fields[5].split("_")
            times = np.fromstring(fields[6], sep="_", dtype=np.int64)
            assert len(candidates) == len(labels) and np.array_equal(raw_labels, labels)
            assert set(labels).issubset({0, 1, 2, 3}) and len(candidates) > 0
            assert len(histories) == len(history_queries) == len(history_actions)
            assert len(times) == len(histories) + 1 and times[0] == 0 and np.all(times >= 0)
            candidate_buffer.frombytes(candidates.tobytes())
            history_buffer.frombytes(histories.tobytes())
            label_buffer.frombytes(labels.tobytes())
            gap_buffer.frombytes(times[1:-1].tobytes())  # exclude initial placeholder and final test gap
            for q, action in zip(history_queries, history_actions):
                assert action in ("CLICK", "CART", "ORD", "FLW")
                actions[action] += 1
                queryless_actions[action] += q == "-1"
                text_observation("past query (non-placeholder)", q)
            terms = tokenize(fields[0])
            query_vocabulary.update(terms)
            raw_queries[fields[0]] += 1
            text_observation("current query", fields[0])
            positive = labels > 0
            has_positive = bool(positive.any())
            positive_records += has_positive
            if has_positive:
                positive_prefix_records += not np.any(positive[1:] & ~positive[:-1])
            label_session_counts += [np.any(labels == grade) for grade in range(4)]
            threshold_session_counts += [True, np.any(labels >= 1), np.any(labels >= 2), np.any(labels == 3)]
            session_rows.append((len(candidates), len(histories), int(times[-1]), int(times[1:-1].sum()),
                                 int((labels > 0).sum()), int((labels == 3).sum())))
            if (session_id + 1) % 50000 == 0:
                print(f"Behaviour records summarised: {session_id + 1:,}", flush=True)
    sessions = pd.DataFrame(session_rows, columns=["slate_len", "history_len", "time_since_last_raw", "history_span_raw", "positive_candidates", "purchase_candidates"])
    cand_ids, hist_ids = np.frombuffer(candidate_buffer, dtype=np.int64), np.frombuffer(history_buffer, dtype=np.int64)
    labels, gaps = np.frombuffer(label_buffer, dtype=np.int8), np.frombuffer(gap_buffer, dtype=np.int64)
    n_records, n_candidates, n_histories = len(sessions), len(cand_ids), len(hist_ids)
    candidate_ids, exposures = np.unique(cand_ids, return_counts=True)
    historical_ids, interactions = np.unique(hist_ids, return_counts=True)
    processed = Path(processed)
    processed.mkdir(parents=True, exist_ok=True)
    np.save(processed / "eda_history_item_ids.npy", historical_ids)
    np.save(processed / "eda_history_item_counts.npy", interactions)
    catalog_buffer, category_buffer = array("q"), array("q")
    field_counts = {field: Counter() for field in ["brand_id", "shop_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4", "brand_name"]}
    missing = Counter()
    with Path(catalogue).open(encoding="utf-8") as handle:
        metadata_header = next(handle).rstrip("\r\n").split("\t")
        assert len(metadata_header) == 13 and metadata_header[0] == "wid"
        for i, line in enumerate(handle):
            fields = line.rstrip("\r\n").split("\t")
            assert len(fields) == 13
            catalog_buffer.append(int(fields[0]))
            category_buffer.append(int(fields[4]))
            for field, position in [("brand_id", 2), ("shop_id", 12), ("cate_id_1", 4), ("cate_id_2", 6), ("cate_id_3", 8), ("cate_id_4", 10), ("brand_name", 3)]:
                value = fields[position]
                if value in ("", "-1"):
                    missing[field] += 1
                else:
                    field_counts[field][value] += 1
            for field, position in [("product title", 1), ("brand name (nonempty)", 3), ("category L3 name", 9)]:
                text_observation(field, fields[position])
                if fields[position] in ("", "-1"):
                    missing[field] += 1
            if (i + 1) % 4000000 == 0:
                print(f"Catalog rows summarised: {i + 1:,}", flush=True)
    catalog_ids, category_ids = np.frombuffer(catalog_buffer, dtype=np.int64), np.frombuffer(category_buffer, dtype=np.int64)
    catalog_order = np.argsort(catalog_ids)
    sorted_catalog = catalog_ids[catalog_order]
    assert np.all(sorted_catalog[1:] > sorted_catalog[:-1]), "Duplicate catalog IDs"
    all_ids = np.union1d(candidate_ids, historical_ids)
    cold = ~members(historical_ids, candidate_ids)
    cold_occurrence_share = exposures[cold].sum() / n_candidates
    catalog_present = members(sorted_catalog, all_ids)
    n_missing_behavior_ids = int((~catalog_present).sum())
    catalog_candidate_mask = members(sorted_catalog, candidate_ids)
    candidate_catalog_share = exposures[catalog_candidate_mask].sum() / n_candidates
    history_catalog_mask = members(sorted_catalog, historical_ids)
    history_catalog_share = interactions[history_catalog_mask].sum() / n_histories
    label_counts = np.bincount(labels, minlength=4)
    label_names = ["No recorded interaction (0)", "Click label (1)", "Cart label (2)", "Purchase label (3)"]
    label_distribution = pd.DataFrame({"candidate occurrences": label_counts, "share of candidates": label_counts / n_candidates,
                                       "searches with this label": label_session_counts, "share of searches": label_session_counts / n_records}, index=label_names)
    threshold_counts = [n_candidates, (labels > 0).sum(), (labels >= 2).sum(), (labels == 3).sum()]
    thresholds = pd.DataFrame({"candidate occurrences": threshold_counts, "share of candidates": np.asarray(threshold_counts) / n_candidates,
                               "searches with this threshold": threshold_session_counts, "share of searches": threshold_session_counts / n_records},
                              index=["All exposed candidates", "Any recorded engagement (>=1)", "Cart or purchase label (>=2)", "Purchase label (=3)"])
    category_counts, category_engaged, category_purchased = Counter(), Counter(), Counter()
    position_counts, position_engaged = np.zeros(30, dtype=np.int64), np.zeros(30, dtype=np.int64)
    # Candidate arrays are read in record order; positions are only used to audit file ordering.
    offsets = np.r_[0, np.cumsum(sessions.slate_len.to_numpy())]
    for start, end in zip(offsets[:-1], offsets[1:]):
        stop = min(30, end - start)
        position_counts[:stop] += 1
        position_engaged[:stop] += labels[start:start + stop] > 0
    for start in range(0, n_candidates, 250000):
        ids, outcomes = cand_ids[start:start + 250000], labels[start:start + 250000]
        locations = np.minimum(np.searchsorted(sorted_catalog, ids), len(sorted_catalog) - 1)
        valid = sorted_catalog[locations] == ids
        categories = np.where(valid, category_ids[catalog_order[locations]], -1)
        values, inverse = np.unique(categories, return_inverse=True)
        count = np.bincount(inverse)
        engaged = np.bincount(inverse, weights=outcomes > 0).astype(np.int64)
        bought = np.bincount(inverse, weights=outcomes == 3).astype(np.int64)
        category_counts.update(dict(zip(values.tolist(), count.tolist())))
        category_engaged.update(dict(zip(values.tolist(), engaged.tolist())))
        category_purchased.update(dict(zip(values.tolist(), bought.tolist())))
    category_rates = pd.DataFrame({"impressions": category_counts, "engaged": category_engaged, "purchased": category_purchased}).fillna(0)
    category_rates["engagement_rate"] = category_rates.engaged / category_rates.impressions
    category_rates["purchase_rate"] = category_rates.purchased / category_rates.impressions
    category_rates.index.name = "cate_id_1"
    category_rates = category_rates.sort_values("impressions", ascending=False)
    def concentration(counts):
        ordered = np.sort(counts)[::-1]
        return [len(counts), counts.mean(), np.median(counts), counts.max(), ordered[:max(1, int(len(counts) * .01))].sum() / counts.sum(),
                ordered[:max(1, int(len(counts) * .20))].sum() / counts.sum(), (counts == 1).mean()]
    popularity = pd.DataFrame({"candidate exposures": concentration(exposures), "history interactions": concentration(interactions)},
                             index=["distinct items", "mean per item", "median per item", "max per item", "share held by top 1%", "share held by top 20%", "share occurring once"])
    action_mix = pd.DataFrame({"actions": actions, "queryless actions": queryless_actions})
    action_mix["share"] = action_mix.actions / n_histories
    action_mix["share queryless within action"] = action_mix["queryless actions"] / action_mix.actions
    categorical = []
    for field, counts in field_counts.items():
        categorical.append({"field": field, "distinct nonmissing": len(counts), "most common values": ", ".join(k for k, _ in counts.most_common(3)),
                            "top value share among nonmissing": max(counts.values()) / sum(counts.values()), "missing": missing[field], "scope": "full catalog"})
    for field, counts in [("history action", actions), ("candidate label", dict(enumerate(label_counts)))]:
        categorical.append({"field": field, "distinct nonmissing": len(counts), "most common values": ", ".join(str(k) for k in counts),
                            "top value share among nonmissing": max(counts.values()) / sum(counts.values()), "missing": 0, "scope": "full behaviour file"})
    categorical = pd.DataFrame(categorical).set_index("field")
    token_lengths = pd.DataFrame({field: describe_frequency(counts) for field, counts in text_counts.items()}).T
    character_lengths = pd.DataFrame({field: describe_frequency(counts) for field, counts in text_chars.items()}).T
    token_lengths["empty / placeholder excluded"] = [excluded_text[field] for field in token_lengths.index]
    # Quantiles need one contiguous gap array; histogramming is chunked to avoid a float copy.
    gap_bins = np.linspace(0, 8, 81)
    gap_hist = np.zeros(len(gap_bins) - 1, dtype=np.int64)
    for start in range(0, len(gaps), 500000):
        gap_hist += np.histogram(np.log10(gaps[start:start + 500000] + 1), bins=gap_bins)[0]
    gap_stats = pd.Series(gaps).describe(percentiles=[.25, .5, .75, .95])
    since_stats = sessions.time_since_last_raw.describe(percentiles=[.25, .5, .75, .95])
    gap_table = pd.DataFrame({"between historical actions (raw units)": gap_stats, "last action to query (raw units)": since_stats})
    gap_table.loc["share equal to zero"] = [(gaps == 0).mean(), sessions.time_since_last_raw.eq(0).mean()]
    overview = pd.DataFrame({"count": {"Users / supplied search records": n_records, "Distinct current query strings": len(raw_queries),
                           "Candidate occurrences": n_candidates, "History actions": n_histories, "Products in catalog": len(catalog_ids),
                           "Distinct candidate products": len(candidate_ids), "Distinct history products": len(historical_ids),
                           "Distinct products in any behaviour": len(all_ids), "Behaviour products missing from catalog": n_missing_behavior_ids,
                           "Distinct brands": len(field_counts["brand_id"]), "Distinct shops (nonmissing)": len(field_counts["shop_id"]),
                           **{f"Distinct category IDs L{k}": len(field_counts[f"cate_id_{k}"]) for k in range(1, 5)},
                           "Current-query term vocabulary": len(query_vocabulary)}})
    feedback = pd.DataFrame({"count": {"Engaged candidate occurrences": int((labels > 0).sum()), "Exposed, no recorded interaction": int((labels == 0).sum()),
                                        "Catalog products absent from all supplied candidate lists": int(len(catalog_ids) - catalog_candidate_mask.sum()), "Positive historical actions": n_histories}})
    metadata_coverage = pd.DataFrame({"metadata present share": {"Candidate occurrences": candidate_catalog_share, "Historical actions": history_catalog_share}})
    return {"sessions": sessions, "overview": overview, "label_distribution": label_distribution, "thresholds": thresholds,
            "action_mix": action_mix, "popularity": popularity, "categorical": categorical, "text_tokens": token_lengths, "text_chars": character_lengths,
            "gap_table": gap_table, "gap_histogram": (gap_bins, gap_hist), "category_rates": category_rates, "feedback": feedback,
            "metadata_coverage": metadata_coverage, "lorenz": {"exposures": compact_lorenz(exposures), "history": compact_lorenz(interactions)},
            "position_rate": pd.Series(position_engaged / position_counts, index=np.arange(1, 31)),
            "positive_prefix_share": positive_prefix_records / positive_records, "all_zero_records": n_records - positive_records,
            "cold_distinct_share": float(cold.mean()), "cold_occurrence_share": float(cold_occurrence_share),
            "candidate_count": n_candidates, "history_count": n_histories, "catalog_count": len(catalog_ids)}


def build_sample_features(behaviour, catalogue, processed, sample_ids):
    """Example features on complete sampled records (all candidates and history), with the shared match definitions."""
    selected = set(np.asarray(sample_ids).tolist())
    records, needed = [], set()
    with Path(behaviour).open(encoding="utf-8") as handle:
        next(handle)
        for session_id, line in enumerate(handle):
            if session_id not in selected:
                continue
            fields = line.rstrip("\r\n").split("\t")
            candidates, past = fields[1].split("_"), fields[4].split("_")
            queries, actions = fields[3].split("_"), fields[5].split("_")
            labels = np.asarray([int(float(v)) for v in fields[2].split("_")], dtype=np.int8)
            times = np.fromstring(fields[6], sep="_", dtype=np.int64)
            assert len(candidates) == len(labels) and len(past) == len(queries) == len(actions)
            assert len(times) == len(past) + 1
            records.append((session_id, fields[0], candidates, labels, past, queries, actions, times))
            needed.update(candidates); needed.update(past)
    metadata, names = {}, {}
    candidate_ids = {wid for record in records for wid in record[2]}
    with Path(catalogue).open(encoding="utf-8") as handle:
        next(handle)
        for line in handle:
            wid = line.split("\t", 1)[0]
            if wid not in needed:
                continue
            fields = line.rstrip("\r\n").split("\t")
            metadata[wid] = tuple(int(fields[j]) if fields[j] not in ("", "-1") else None for j in (2, 12, 4, 6, 8, 10))
            if wid in candidate_ids:
                names[wid] = (fields[1], fields[3], fields[9])
    hist_ids = np.load(Path(processed) / "eda_history_item_ids.npy", mmap_mode="r")
    hist_counts = np.load(Path(processed) / "eda_history_item_counts.npy", mmap_mode="r")
    feature_parts, sessions = [], []
    for session_id, query, candidates, labels, past, queries, actions, times in records:
        query_terms = token_set(query)
        has_query = np.asarray([q != "-1" for q in queries])
        scores, known, _, _ = match_counts(candidates, past, has_query, metadata)
        row = {"session_id": session_id, "hist_len": len(past), "hist_share_no_query": (~has_query).mean(),
               "hist_metadata_coverage": sum(wid in metadata for wid in past) / len(past),
               "time_since_last_raw": times[-1], "time_since_last_h_assumed": times[-1] / 3600,
               "history_span_days_assumed": times[1:-1].sum() / 86400, "query_len_chars": len(query),
               "query_len_terms": len(tokenize(query)), "query_distinct_terms": len(query_terms), "slate_len": len(candidates)}
        for action in ("CLICK", "CART", "ORD", "FLW"):
            row[f"hist_n_{action.lower()}"] = actions.count(action)
        similarities = [overlap_features(query_terms, token_set(q))[2] for q in queries if q != "-1"]
        row["history_query_max_jaccard"] = max(similarities, default=0)
        row["history_query_any_overlap"] = row["history_query_max_jaccard"] > 0
        row["history_query_same_terms"] = any(token_set(q) == query_terms for q in queries if q != "-1")
        sessions.append(row)
        candidate_array = np.asarray(candidates, dtype=np.int64)
        indices = np.minimum(np.searchsorted(hist_ids, candidate_array), len(hist_ids) - 1)
        popularity = np.where(hist_ids[indices] == candidate_array, hist_counts[indices], 0)
        ordered_products = {p for p, a in zip(past, actions) if a == "ORD"}
        pair = pd.DataFrame({"session_id": session_id, "label": labels, "item_hist_popularity_eda": popularity,
                             "item_cold_eda": popularity == 0, "item_in_catalogue": [wid in metadata for wid in candidates],
                             "item_seen_before": scores["all"][:, 0] > 0,
                             "item_ordered_before": [wid in ordered_products for wid in candidates]})
        for source in SOURCES:
            for j, level in enumerate(LEVELS):
                pair[f"aff_{source}_{level}"] = np.where(known[:, j], scores[source][:, j], np.nan)
        text_rows = []
        for wid in candidates:
            if wid not in names:
                text_rows.append((np.nan,) * 5)
                continue
            title, brand, category = (token_set(text) for text in names[wid])
            shared, coverage, jaccard = overlap_features(query_terms, title)
            text_rows.append((shared if title else np.nan, coverage if title else np.nan, jaccard if title else np.nan,
                              bool(query_terms & brand) if brand else np.nan, bool(query_terms & category) if category else np.nan))
        pair[["query_title_shared_terms", "query_title_coverage", "query_title_jaccard", "query_brand_match", "query_cate3_match"]] = pd.DataFrame(text_rows)
        feature_parts.append(pair)
    return pd.DataFrame(sessions).set_index("session_id"), pd.concat(feature_parts, ignore_index=True)

### 2.1 Stream the raw files

Item IDs are stored as compact numeric arrays for exact entity counts, avoiding full text tables and candidate-feature matrices. The history-frequency arrays saved to `processed/` are used for description only.

In [ ]:
full = build_full_profile(paths.behaviour, paths.catalogue, paths.processed)
record_summary = full["sessions"]
n_sessions = len(record_summary)
print(f"Records: {n_sessions:,}; candidates: {full['candidate_count']:,}; history actions: {full['history_count']:,}")
print("Raw list lengths, label values, time alignment and catalog ID uniqueness passed.")

In [ ]:
display(save_table(full["metadata_coverage"], "2_1_metadata_coverage_full", FULL_SCOPE))
print(f"Catalog products: {full['catalog_count']:,}")

## 3. Data overview and sampling scheme

### 3.1 Source and entity counts

- **Source:** Liu et al. (2023), *JDsearch: A Personalized Product Search Dataset with Real Queries and Full Interactions*, SIGIR. [Paper](https://arxiv.org/html/2305.14810); [format README](https://github.com/rucliujn/JDsearch#dataset-files).
- **Record unit:** the source describes one sampled user's last test query and supplied history per row. No persistent user-ID field is released, so rows provide the grouping; we cannot link additional searches to an account ourselves.
- **Anonymisation:** query and metadata text contain term IDs separated by `\x18`; list entries use `_`. Original words are unavailable, but token boundaries are recoverable.

In [ ]:
overview = full["overview"].copy()
overview.loc["Distinct products across catalog and behaviour", "count"] = (
    int(overview.loc["Products in catalog", "count"]) +
    int(overview.loc["Behaviour products missing from catalog", "count"])
)
display(save_table(overview, "2_1_entity_counts", FULL_SCOPE))

### 3.2 Collection period and relative time

The paper (section 3.1) documents history collection from **18 October 2021 to 17 October 2022**, with the last (test) queries on **17 October 2022**. The released file provides relative gaps, not absolute timestamps, so a cross-user chronological split cannot be reconstructed.

The README specifies an initial zero, one gap per transition and a final gap to the current query; length and initial-zero checks run on every record. Hours and days below **assume seconds** (the released schema does not state the unit).

In [ ]:
span_days = record_summary.history_span_raw / 86400
span = span_days.describe(percentiles=[.25, .5, .75, .95]).rename("history span in days (seconds assumed)")
display(save_table(span.to_frame(), "2_1_history_span_days_assumed", FULL_SCOPE))
print(f"Derived spans over 365 days: {(span_days > 365).sum():,} ({(span_days > 365).mean():.4%})")
print("Spans exceeding the documented year need a time-unit/logging audit before calendar interpretation.")

### 3.3 Sampling scheme

Raw summaries cover the full files. Engineered features use a **uniform sample of 5,000 whole search records**, with seed 4222; every candidate and historical action in each selected row is retained. Metadata is looked up against the complete catalog for both history and candidates, avoiding artificial missingness.

This is an exploratory feature sample, not a model split (the split is made in notebook 02, which keeps these 5,000 records out of validation and test). The table below compares the sample with the full file; a random sample does not remove the source dataset's selection bias.

In [ ]:
rng = np.random.default_rng(SEED)
sample_ids = np.sort(rng.choice(n_sessions, min(EDA_SAMPLE_SIZE, n_sessions), replace=False))
assert np.array_equal(sample_ids, jd.explored_ids(n_sessions))  # the same draw notebook 02 keeps out of validation and test
print(f"{len(sample_ids):,} sampled records; all candidates and history retained")

In [ ]:
def profile(rows):
    return {"records": len(rows), "candidate occurrences": rows.slate_len.sum(),
            "mean slate length": rows.slate_len.mean(), "median history length": rows.history_len.median(),
            "candidate engagement share": rows.positive_candidates.sum() / rows.slate_len.sum(),
            "candidate purchase-label share": rows.purchase_candidates.sum() / rows.slate_len.sum()}
representativeness = pd.DataFrame({"full file": profile(record_summary),
                                  "5,000-record sample": profile(record_summary.iloc[sample_ids])})
display(save_table(representativeness, "2_1_sample_vs_full", "full file compared with uniform feature sample"))

## 4. Impression, exposure and interaction dynamics

### 4.1 Impression slates

In [ ]:
slate_len = record_summary.slate_len
slate_stats = slate_len.describe(percentiles=[.05, .25, .5, .75, .95]).rename("candidates per search")
display(save_table(slate_stats.to_frame(), "2_2_slate_length", FULL_SCOPE))
fig, ax = plt.subplots()
ax.hist(slate_len, bins=range(0, 205, 5), color=BLUE, edgecolor="white")
ax.axvline(slate_len.median(), color=ORANGE, linestyle="--", label=f"median {slate_len.median():.0f}")
ax.set(title="Products supplied per search", xlabel="candidates", ylabel="search records")
ax.legend(frameon=False)
save_fig(fig, "2_2_slate_length")

### 4.2 Interaction grades and thresholds

Each candidate has one label: 0 no recorded interaction, 1 click, 2 cart, 3 purchase. Show both exclusive label volumes and cumulative grade thresholds. These thresholds do **not** reveal the sequence of events: a purchase label does not establish an observed click-to-cart-to-purchase path. Ratios between threshold counts must not be called event conversion rates.

In [ ]:
label_distribution = full["label_distribution"]
thresholds = full["thresholds"]
display(save_table(label_distribution, "2_2_interaction_labels", FULL_SCOPE))
display(save_table(thresholds, "2_2_interaction_thresholds", FULL_SCOPE))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, column, title in [(axes[0], "share of candidates", "Candidate grade thresholds"),
                          (axes[1], "share of searches", "Searches containing each grade threshold")]:
    values = thresholds[column].iloc[1:]
    ax.bar(["Any engagement", "Cart/purchase", "Purchase"], values, color=[BLUE, AQUA, ORANGE])
    for x, value in enumerate(values):
        ax.annotate(f"{value:.2%}", (x, value), ha="center", va="bottom")
    ax.set(title=title)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
plt.tight_layout()
save_fig(fig, "2_2_interaction_thresholds")

**Interpretation.** Most candidates have no recorded interaction; accuracy alone would reward predicting zero. The supplied searches usually contain an engaged product, supporting evaluation of how it is ranked. This reflects the dataset's selected last-search records, not the probability that an arbitrary JD search converts. Purchase labels are observed outcomes, not measured revenue or evidence of the intervening interaction sequence.

### 4.3 Candidate-file order audit

The full-file check below verifies whether positive candidates are stored before negatives. This ordering cannot serve as a reliable recorded display rank, so a CTR-by-display-position analysis is unavailable. The chart describes a **file-order artifact**, not user position bias.

Consequence for the pipeline: no model receives a candidate position, candidates are shuffled within each search before any training (notebooks 02 and 03), and the evaluation breaks ties at random.

In [ ]:
print(f"Positive-containing records with every positive before every negative: {full['positive_prefix_share']:.2%}")
print(f"Records without positives: {full['all_zero_records']:,} ({full['all_zero_records'] / n_sessions:.2%})")
assert full["positive_prefix_share"] == 1.0
fig, ax = plt.subplots()
ax.plot(full["position_rate"].index, full["position_rate"].values, color=BLUE, marker="o", markersize=3)
ax.set(title="Recorded engagement by file position (not display rank)", xlabel="file position", ylabel="engagement share")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
save_fig(fig, "2_2_position_artifact")

### 4.4 Feedback semantics

| Signal | Meaning |
|---|---|
| Candidate label 1–3 | Exposed with a recorded click/cart/purchase outcome |
| Candidate label 0 | Exposed without a recorded interaction; usable implicit negative, not an explicit dislike |
| Product outside this record's slate | Unobserved for this search; cannot infer exposure or preference |
| Historical action | A past recorded positive action; no historical exposure negatives are supplied |

A product absent from all supplied candidate lists may still have historical interactions. Neither absence from the current slate nor absence from every supplied slate proves it was never shown anywhere on JD. All models rank the supplied (exposed) candidates.

In [ ]:
display(save_table(full["feedback"], "2_2_feedback_semantics", FULL_SCOPE))

### 4.5 Sparsity: history length, item popularity, cold start

In [ ]:
history_len = record_summary.history_len
hist_stats = history_len.describe(percentiles=[.05, .25, .5, .75, .95, .99]).rename("history actions per record")
display(save_table(hist_stats.to_frame(), "2_2_history_length", FULL_SCOPE))
display(save_table(full["action_mix"], "2_2_history_action_mix", FULL_SCOPE))
fig, ax = plt.subplots()
ax.hist(np.log10(history_len + 1), bins=60, color=BLUE, edgecolor="white")
ax.set(title="Supplied history lengths", xlabel="log10(history actions + 1)", ylabel="records")
save_fig(fig, "2_2_history_length")

In [ ]:
display(save_table(full["popularity"], "2_2_item_popularity", FULL_SCOPE))
cold_stats = pd.DataFrame({"share": {"Distinct candidate IDs absent from all histories": full["cold_distinct_share"],
                                    "Candidate occurrences absent from all histories": full["cold_occurrence_share"]}})
display(save_table(cold_stats, "2_2_cold_item_coverage", FULL_SCOPE))
print("These are full-data descriptive history-absence measures, not coldness relative to a future training split.")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for name, color in [("exposures", BLUE), ("history", ORANGE)]:
    x, y = full["lorenz"][name]
    ax.plot(x, y, color=color, label=name)
ax.plot([0, 1], [0, 1], color=MUTED, linestyle="--")
ax.set(title="Item frequency concentration", xlabel="share of items, most frequent first", ylabel="share of volume")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1))
ax.legend(frameon=False)
save_fig(fig, "2_2_item_popularity")

### 4.6 Temporal patterns: time between actions

In [ ]:
display(save_table(full["gap_table"], "2_2_time_gaps_raw", FULL_SCOPE))
fig, ax = plt.subplots()
bins, counts = full["gap_histogram"]
width = np.diff(bins)
ax.stairs(counts / counts.sum() / width, bins, color=BLUE, label="between historical actions")
since = record_summary.time_since_last_raw.to_numpy()
since_counts = np.histogram(np.log10(since + 1), bins=bins)[0]
ax.stairs(since_counts / since_counts.sum() / width, bins, color=ORANGE, label="last action to query")
ax.set(title="Relative time gaps (raw units)", xlabel="log10(gap + 1)", ylabel="density")
ax.legend(frameon=False)
save_fig(fig, "2_2_time_gaps_raw")

**Interpretation.** The intervals include short bursts and long gaps, which motivates recency features (notebook 03) but does not establish that recent actions are more predictive. Initial zero placeholders are excluded from between-action summaries; genuine zero transition gaps remain.

### 4.7 Observed outcomes by broad category

Compare label shares by the candidate's L1 category. Differences are descriptive associations affected by exposure and query mix; they do not establish causal category effects or whether category depth improves within-query ranking. ID `-1` in this table denotes unavailable catalog metadata.

In [ ]:
by_cate = full["category_rates"]
top_cate = by_cate.loc[by_cate.index != -1].head(15)
display(save_table(top_cate, "2_2_category_engagement", FULL_SCOPE))
fig, ax = plt.subplots(figsize=(9, 5))
plot = top_cate.iloc[::-1]
ax.barh(plot.index.astype(str), plot.purchase_rate, color=BLUE)
ax.set(title="Purchase-label share in frequent L1 categories", xlabel="share of supplied candidates with label 3", ylabel="L1 category ID")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1, decimals=2))
save_fig(fig, "2_2_category_engagement")

**Interpretation.** Graded labels distinguish recorded clicks, carts and purchases. Giving purchases a greater ranking gain is an evaluation choice; it does not make the target a purchase probability or a monetary value.

## 5. Feature dictionary and descriptive statistics

### 5.1 Raw fields

In [ ]:
raw_dictionary = pd.DataFrame(
    [
        ("query", "user", "Current search query, anonymised term IDs", "text (separated term IDs)"),
        ("candidate_wid_list", "user", "Products shown for the query (the slate)", "list of item IDs"),
        ("candidate_label_list", "user", "Interaction per candidate: 0 none, 1 click, 2 cart, 3 purchase", "list of ordinal labels"),
        ("history_qry_list", "user", "Query behind each past action; -1 = not from search", "list of text / -1"),
        ("history_wid_list", "user", "Product of each past action", "list of item IDs"),
        ("history_type_list", "user", "Action type: CLICK, CART, FLW (follow), ORD (purchase)", "list of categorical"),
        ("history_time_list", "user", "Gap to previous action; final value = gap to current query", "list of nonnegative int gaps (raw unit; seconds assumed)"),
        ("wid", "product", "Product ID", "categorical ID"),
        ("name", "product", "Product title, anonymised term IDs", "text (separated term IDs)"),
        ("brand_id / brand_name", "product", "Brand", "categorical ID / text"),
        ("cate_id_1..4 / cate_name_1..4", "product", "Four-level category hierarchy", "categorical ID / text"),
        ("shop_id", "product", "Seller shop", "categorical ID"),
    ],
    columns=["field", "file", "description", "type / unit"],
).set_index("field")
display(save_table(raw_dictionary, "2_3_raw_fields", "released raw-field definitions"))

In [ ]:
display(save_table(full["categorical"], "2_3_categorical_fields", FULL_SCOPE))

In [ ]:
# Exact token/character length distributions cover each entire applicable file.
# Query-less -1 placeholders and empty metadata text are excluded and counted separately.
display(save_table(full["text_tokens"], "2_3_text_lengths_tokens", FULL_SCOPE))
display(save_table(full["text_chars"], "2_3_text_lengths_chars", FULL_SCOPE))

### 5.2 Sampled feature dictionary and statistics

Example features on the uniform 5,000-record sample. Historical inputs precede the current query; candidate labels are used only for descriptive comparisons. Product / brand / shop and L1-L4 **path** counts use the match definitions of `jdsearch.match_counts`, split into all / search-linked / query-less history. Missing candidate metadata gives unavailable features (`NaN`), never a match to other missing values.

Population history counts and cold flags have an `_eda` suffix: they use the full file here **for description only**. The model features (notebook 03) derive population statistics from training records only, and supervised encodings out-of-fold. A zero history count can also reflect missing historical metadata.

In [ ]:
session_features, sample_pair = build_sample_features(paths.behaviour, paths.catalogue, paths.processed, sample_ids)
assert session_features.index.to_numpy().tolist() == sample_ids.tolist()
assert len(sample_pair) == record_summary.iloc[sample_ids].slate_len.sum()
assert sample_pair.label.gt(0).sum() == record_summary.iloc[sample_ids].positive_candidates.sum()
assert sample_pair.label.eq(3).sum() == record_summary.iloc[sample_ids].purchase_candidates.sum()
print(f"Sample records: {len(session_features):,}; full sampled candidate occurrences: {len(sample_pair):,}")
print("Category features match full ancestor paths.")

**Query features:** title shared-term count, query coverage and Jaccard similarity, plus any-term brand/category-name overlap. These compare complete `\x18`-separated IDs. Empty/unavailable names yield missing text features. An overlap with a brand name is lexical evidence; it does not establish a brand-intent query type. Current/history query overlap ignores token order; repeated historical queries can be multiple actions under one issued query.

In [ ]:
query_columns = ["query_title_shared_terms", "query_title_coverage", "query_title_jaccard", "query_brand_match", "query_cate3_match"]
display(numeric_stats(sample_pair[query_columns].astype(float)))

In [ ]:
dictionary_rows = []
for feature in session_features.columns:
    description = {
        "hist_len": "Supplied past action count", "hist_share_no_query": "Fraction of supplied actions with query -1",
        "hist_metadata_coverage": "Fraction of historical actions whose product is in the catalog",
        "time_since_last_raw": "Final gap to test query, raw units", "time_since_last_h_assumed": "Final gap / 3600; seconds assumed",
        "history_span_days_assumed": "Historical transition gaps / 86400; seconds assumed",
        "query_len_chars": "Characters in current anonymised query", "query_len_terms": "Term occurrences in current query",
        "query_distinct_terms": "Distinct query term IDs", "slate_len": "Supplied candidates for the query",
        "history_query_max_jaccard": "Largest current/past query distinct-term Jaccard overlap",
        "history_query_any_overlap": "Any historical query shares a current-query term",
        "history_query_same_terms": "Any historical query has the same distinct term set",
    }.get(feature, "Historical action count by recorded type")
    unit = {
        "time_since_last_raw": "raw interval units (nonnegative integer)",
        "time_since_last_h_assumed": "hours (seconds assumed)",
        "history_span_days_assumed": "days (seconds assumed)",
        "query_len_chars": "characters",
    }.get(feature, "binary" if feature.endswith(("any_overlap", "same_terms")) else "ratio 0-1" if feature in ["hist_share_no_query", "hist_metadata_coverage", "history_query_max_jaccard"] else "count")
    dictionary_rows.append((feature, "search record", description, unit, "computed on 5,000 records"))
for feature in sample_pair.columns:
    if feature in ["session_id", "label"]:
        continue
    if feature.startswith("aff_"):
        description = "Prior action count for candidate's product/attribute/path in the named history source; NaN if candidate attribute unavailable"
        unit = "count"
    else:
        description = {
            "item_hist_popularity_eda": "Candidate action count in all histories; descriptive only, train-only in models",
            "item_cold_eda": "Candidate absent from all histories; descriptive, not training-split coldness",
            "item_in_catalogue": "Catalog contains candidate metadata", "item_seen_before": "Exact candidate ID occurs in own history",
            "item_ordered_before": "Own history contains ORD for this exact candidate",
            "query_title_shared_terms": "Number of shared distinct query/title term IDs",
            "query_title_coverage": "Shared distinct query/title terms divided by distinct query terms",
            "query_title_jaccard": "Shared distinct query/title terms divided by their union",
            "query_brand_match": "Any shared query/brand-name term; missing if name unavailable",
            "query_cate3_match": "Any shared query/L3-category-name term; missing if name unavailable",
        }[feature]
        unit = "count" if feature in ["item_hist_popularity_eda", "query_title_shared_terms"] else "ratio" if feature in ["query_title_coverage", "query_title_jaccard"] else "binary"
    dictionary_rows.append((feature, "candidate", description, unit, "computed on complete sampled slates"))
engineered_dictionary = pd.DataFrame(dictionary_rows, columns=["feature", "level", "description", "type/unit", "status"]).set_index("feature")
display(save_table(engineered_dictionary, "2_3_engineered_dictionary", SAMPLE_SCOPE))

In [ ]:
session_stats = numeric_stats(session_features.astype(float))
display(save_table(session_stats, "2_3_session_feature_stats", SAMPLE_SCOPE))

In [ ]:
pair_columns = [c for c in sample_pair if c not in ["session_id", "label"]]
pair_stats = numeric_stats(sample_pair[pair_columns].astype(float))
engaged = sample_pair.label > 0
pair_stats["mean among engaged"] = sample_pair.loc[engaged, pair_columns].astype(float).mean()
pair_stats["mean among non-engaged"] = sample_pair.loc[~engaged, pair_columns].astype(float).mean()
pair_stats["ratio of feature means"] = pair_stats["mean among engaged"] / pair_stats["mean among non-engaged"]
display(save_table(pair_stats, "2_3_pair_feature_stats", SAMPLE_SCOPE))

**Interpretation.** The ratio compares feature means between engaged and non-engaged candidates. It is **not** an engagement-rate multiplier or evidence of a model gain: pooled differences can reflect query mix, history length and exposure. Exact-product history is strong but rare evidence; broader shop, brand and category matches are common but weaker. Which signals help a trained ranker is tested in notebook 05.

In [ ]:
ratios = pair_stats["ratio of feature means"].replace([np.inf, -np.inf], np.nan).dropna()
ratios = ratios.loc[(ratios > 0) & ~ratios.index.isin(["item_in_catalogue"])]
fig, ax = plt.subplots(figsize=(9, 8))
ratios = ratios.sort_values()
ax.barh(ratios.index, ratios.values, color=BLUE)
ax.axvline(1, color=MUTED, linestyle="--")
ax.set_xscale("log")
ax.set(title="Descriptive feature-mean ratios (5,000 records)", xlabel="mean among engaged / mean among non-engaged")
save_fig(fig, "2_3_feature_mean_ratios")

## 6. Limitations and consequences for the pipeline

1. **File order encodes outcomes.** Positives precede negatives, so display-rank CTR is unavailable; no position feature is used, candidates are shuffled within each search and ties are broken at random.
2. **Record grouping and collection period.** Each row is one user's last query; no persistent user ID or absolute timestamps are released, so a cross-user temporal split is impossible. The split is by whole search record (notebook 02).
3. **Time interpretation.** Raw gaps and alignment are verified; hours and days assume seconds, and some derived history spans exceed the documented year.
4. **Anonymised text and queries.** Term boundaries are recoverable, meanings are not; token overlap is lexical evidence only, and repeated history-query entries may be actions from one search.
5. **Observed labels and selection.** Grade thresholds are not observed funnel transitions. Nearly all supplied searches have an engaged candidate, so these labels do not give general JD conversion rates; searches without one are excluded from evaluation (notebook 02).
6. **Missing metadata and sparse matches.** Unknown values never match each other; missing historical metadata can hide preferences. Full-data absence from history differs from coldness relative to a training split.
7. **Category identity.** Some L3 / L4 IDs have several parents, so category matches use full ancestor paths.
8. **Population statistics.** Raw summaries here are full-data and descriptive; model features compute population statistics on training records only (notebook 03).

In [ ]:
print("Saved figures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(f"  {path.relative_to(OUT)}")
print("Saved tables:")
for path in sorted(TABLE_DIR.glob("*.csv")):
    print(f"  {path.relative_to(OUT)}")